<!-- notebook:description -->
# Notebook: RQ2 — Inconsistency Study (Oracle ↔ Tests)

**Rôle** : analyser les rapports JSON produits par l’expérimentation RQ2 pour mesurer et caractériser les incohérences entre les oracles dérivés et les tests générés (scripts Java / Gherkin).

**Objectif** : construire des tables d’analyse (résultats par endpoint, détails des incohérences), produire des visualisations, et exporter des artefacts (CSV/JSON/figures) pour le reporting scientifique.

*Notebook basé sur données réelles (rapports JSON générés par l’expérimentation RQ2). Pas de simulation.*

## Énoncé de la question de recherche (RQ2)
- **RQ2 — Gap Detection and Reduction** : Dans quelle mesure une architecture multi-agent permet-elle la détection et la réduction des incohérences entre les tests induits et les scripts générés ?

> Notebook basé sur données réelles (rapports JSON générés par l’expérimentation RQ2).

## 1. Setup et Imports

Cette cellule initialise l’environnement d’analyse : imports Python, configuration d’affichage, ajout du chemin du dépôt pour importer les utilitaires de reporting, et création des répertoires de sortie (figures / résultats).

In [ ]:
# Setup & imports (RQ2)
# - Importe les dépendances et configure le style des figures pour le reporting.
# - Ajoute la racine du dépôt au PYTHONPATH pour importer `experiments.*`.

# Imports standard
import sys
import json
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List

# Imports data science
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 200)

# Ensure repo root is importable (enables `import experiments.*`)
PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiments.reporting_utils import PublicationStyle, apply_publication_style, save_figure

apply_publication_style(
    PublicationStyle(
        seaborn_style="whitegrid",
        seaborn_context="paper",
        font_scale=1.1,
        palette="colorblind",
    )
)

EXPERIMENTS_RESULTS_DIR = PROJECT_ROOT / "experiments" / "results"
EXPERIMENTS_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_DIR = EXPERIMENTS_RESULTS_DIR / "figures" / "rq2"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, stem: str):
    """Save a figure to FIGURES_DIR as PNG+PDF (stem without extension)."""
    return save_figure(fig, FIGURES_DIR / stem, formats=["png", "pdf"], dpi=300)

print("✓ Imports réussis")
print(f"✓ Racine du projet: {PROJECT_ROOT}")

## 2. Configuration de l'Expérience

Cette cellule définit les chemins d’entrée/sortie de l’expérience (répertoires de résultats et de données) afin de standardiser l’accès aux fichiers et d’éviter les erreurs liées au chemin courant.

In [ ]:
# Configuration des chemins
# - Définit les répertoires utilisés par l’analyse (résultats et données).
# - Crée les dossiers au besoin pour éviter les erreurs d’I/O.

# Configuration des chemins
RESULTS_DIR = PROJECT_ROOT / "experiments" / "results" / "rq2"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Répertoire résultats: {RESULTS_DIR}")
print(f"✓ Répertoire données: {DATA_DIR}")

## 3. Chargement des Résultats (Réel)

Cette cellule transforme le rapport JSON RQ2 en deux tables : une table “par endpoint” (scores, couvertures, compteurs) et une table “détaillée” listant chaque incohérence (sévérité, type, catégorie, recommandation).

In [ ]:
# Chargement du rapport et construction des tables d’analyse
# - Transforme la structure JSON en DataFrames: `results_df` (par endpoint) et `inconsistencies_df` (détaillé).
# - Normalise les compteurs par sévérité et extrait quelques champs utiles pour le reporting.

# Construire des DataFrames à partir du rapport JSON (réel)
endpoint_results: List[Dict[str, Any]] = report.get("endpoint_results", [])

rows: List[Dict[str, Any]] = []
inconsistency_rows: List[Dict[str, Any]] = []

for er in endpoint_results:
    counts = er.get("inconsistency_counts", {})
    flags = er.get("quality_flags", {})
    exec_meta = er.get("execution_metadata", {})

    critical = int(counts.get("critical", 0))
    major = int(counts.get("major", 0))
    minor = int(counts.get("minor", 0))
    info = int(counts.get("info", 0))
    total_inconsistencies = critical + major + minor + info

    rows.append({
        "endpoint_id": er.get("endpoint_id"),
        "endpoint_name": er.get("endpoint_name"),
        "llm_model": er.get("llm_model"),
        "coherence_score": float(er.get("coherence_score", 0.0)),
        "java_coverage_ratio": float(er.get("java_coverage_ratio", 0.0)),
        "gherkin_coverage_ratio": float(er.get("gherkin_coverage_ratio", 0.0)),
        "critical_count": critical,
        "major_count": major,
        "minor_count": minor,
        "info_count": info,
        "total_inconsistencies": total_inconsistencies,
        "missing_validations": int(counts.get("missing_validations", 0)),
        "extra_validations": int(counts.get("extra_validations", 0)),
        "incorrect_implementations": int(counts.get("incorrect_implementations", 0)),
        "passes_threshold": bool(flags.get("passes_threshold", False)),
        "has_critical_issues": bool(flags.get("has_critical_issues", False)),
        "has_major_issues": bool(flags.get("has_major_issues", False)),
        "generation_time_seconds": float(exec_meta.get("generation_time_seconds", 0.0)),
        "validation_time_seconds": float(exec_meta.get("validation_time_seconds", 0.0)),
        "error_message": exec_meta.get("error_message"),
    })

    incs_by_severity = er.get("inconsistencies", {})
    for severity, incs in incs_by_severity.items():
        for inc in incs:
            inconsistency_rows.append({
                "endpoint_id": er.get("endpoint_id"),
                "endpoint_name": er.get("endpoint_name"),
                "llm_model": er.get("llm_model"),
                "severity": severity,
                "type": inc.get("type"),
                "category": inc.get("category"),
                "field_name": inc.get("field_name"),
                "recommendation": inc.get("recommendation"),
            })

results_df = pd.DataFrame(rows)
inconsistencies_df = pd.DataFrame(inconsistency_rows)

# Alias pratique: plusieurs cellules de visualisation utilisent `df`
df = inconsistencies_df

print("📦 Données chargées:")
print(f"  - Endpoint results: {len(results_df)}")
print(f"  - Inconsistency items: {len(inconsistencies_df)}")
display(results_df.head(5))

## 4. Synthèse des Résultats (Réel)

Cette cellule calcule une synthèse globale et une synthèse par modèle (moyenne/écart-type de cohérence, taux de succès au seuil, incohérences moyennes, couvertures moyennes) pour comparer les modèles sur RQ2.

In [ ]:
# Synthèse globale et par modèle
# - Extrait la config (seuils, modèles) depuis le rapport.
# - Calcule des agrégations par modèle (moyenne cohérence, taux de succès, incohérences moyennes, couvertures).

# Synthèse globale et par modèle
cfg = report.get('config', {})
thresholds = cfg.get('thresholds', {})
min_coherence = thresholds.get('min_coherence_score')

total_endpoints_reported = report.get('total_endpoints')
if total_endpoints_reported is not None:
    total_endpoints = int(total_endpoints_reported)
elif (not results_df.empty) and ('endpoint_id' in results_df.columns):
    total_endpoints = int(results_df['endpoint_id'].nunique())
else:
    total_endpoints = 0

# Déterminer la liste des modèles sans casser si results_df est vide
cfg_models = cfg.get('llm_models', []) or []
if cfg_models:
    models = sorted([str(m) for m in cfg_models])
elif (not results_df.empty) and ('llm_model' in results_df.columns):
    models = sorted(results_df['llm_model'].dropna().astype(str).unique().tolist())
else:
    models = []

print("🔍 Résumé:")
print(f"  Total endpoints (report): {total_endpoints}")
print(f"  Modèles: {models}")
print(f"  Seuil cohérence: {min_coherence}")

if results_df.empty:
    print("⚠ Aucun endpoint_result dans le rapport: analyse limitée.")
elif 'llm_model' not in results_df.columns:
    print("⚠ Colonne 'llm_model' absente dans results_df: impossible de grouper par modèle.")
else:
    # Recalcul passes_threshold si non fourni
    if min_coherence is not None:
        results_df['passes_threshold'] = results_df['coherence_score'] >= float(min_coherence)

    by_model = (
        results_df.groupby('llm_model')
        .agg(
            endpoints=('endpoint_id', 'count'),
            coherence_mean=('coherence_score', 'mean'),
            coherence_std=('coherence_score', 'std'),
            pass_rate=('passes_threshold', 'mean'),
            critical_avg=('critical_count', 'mean'),
            major_avg=('major_count', 'mean'),
            minor_avg=('minor_count', 'mean'),
            java_cov_mean=('java_coverage_ratio', 'mean'),
            gherkin_cov_mean=('gherkin_coverage_ratio', 'mean'),
        )
        .reset_index()
        .sort_values('coherence_mean', ascending=False)
    )
    display(by_model)

## 5. Analyse des Résultats

### 5.1 Distribution des Scores de Cohérence

Cette cellule produit une visualisation exploratoire (histogramme) lorsque la colonne attendue est disponible, afin de comprendre la distribution des scores et repérer d’éventuels outliers.

In [ ]:
# Visualisation exploratoire
# - Produit une figure uniquement si les colonnes attendues sont présentes.
# - Sauvegarde la figure dans le répertoire RQ2 pour le reporting.

# Distribution of incoherence scores (if present)
if "incoherence_score" in df.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(df["incoherence_score"].dropna(), bins=20, kde=True, ax=ax)
    ax.set_title("RQ2 — Incoherence score distribution")
    ax.set_xlabel("Incoherence score")
    ax.set_ylabel("Count")
    fig.tight_layout()

    save_fig(fig, "rq2_incoherence_score_distribution")
    plt.show()

### 5.2 Distribution par Type d'Inconsistance

Cette cellule agrège les mesures par catégorie/type (si les colonnes existent) pour comparer les incohérences selon leur nature, puis sauvegarde la figure dans les formats attendus.

In [ ]:
# Analyse par catégorie/type
# - Détecte automatiquement les colonnes “catégorie/type” et la mesure associée.
# - Agrège puis trace une moyenne par groupe si possible.

# Coherence by category/type (if present)
cat_col = next((c for c in df.columns if str(c).lower() in {"category", "type", "inconsistency_type", "kind"}), None)
val_col = next((c for c in df.columns if "coherence" in str(c).lower() or "incoherence" in str(c).lower()), None)

if cat_col is not None and val_col is not None:
    agg = (df[[cat_col, val_col]].dropna().groupby(cat_col)[val_col].mean().reset_index())
    agg = agg.sort_values(val_col, ascending=False)

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=agg, x=cat_col, y=val_col, ax=ax)
    ax.set_title(f"RQ2 — Mean {val_col} by {cat_col}")
    ax.set_xlabel(cat_col)
    ax.set_ylabel(f"Mean {val_col}")
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()

    safe_cat = str(cat_col).strip().lower().replace(" ", "_")
    safe_val = str(val_col).strip().lower().replace(" ", "_")
    save_fig(fig, f"rq2_mean_{safe_val}_by_{safe_cat}")
    plt.show()

### 5.3 Distribution par Catégorie d'Inconsistance

Cette cellule calcule et visualise la distribution des sévérités (si disponible) pour quantifier le poids relatif des incohérences critiques/majeures/mineures.

In [ ]:
# Distribution des sévérités
# - Compte les occurrences par niveau de sévérité si la colonne existe.
# - Produit une barre de distribution et sauvegarde la figure.

# Severity distribution (if present)
sev_col = next((c for c in df.columns if "severity" in str(c).lower()), None)
if sev_col is not None:
    sev_counts = df[sev_col].value_counts(dropna=False).reset_index()
    sev_counts.columns = ["Severity", "Count"]

    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    sns.barplot(data=sev_counts, x="Severity", y="Count", ax=ax)
    ax.set_title("RQ2 — Severity distribution")
    ax.set_xlabel("Severity")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()

    save_fig(fig, "rq2_severity_distribution")
    plt.show()

### 5.4 Relation Cohérence vs Inconsistances

Cette cellule explore les liens entre variables (ex. couverture) et résultats (cohérence / incohérences), en produisant des agrégations et des graphiques conditionnels selon la disponibilité des colonnes.

In [ ]:
# Analyse couverture (si disponible)
# - Vérifie la présence de colonnes langue/couverture.
# - Agrège la couverture moyenne et produit un graphique.

# Coverage by language (if present)
lang_col = next((c for c in df.columns if str(c).lower() in {"language", "lang"}), None)
cov_col = next((c for c in df.columns if "coverage" in str(c).lower()), None)
if lang_col is not None and cov_col is not None:
    agg = (df[[lang_col, cov_col]].dropna().groupby(lang_col)[cov_col].mean().reset_index())
    agg = agg.sort_values(cov_col, ascending=False)

    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    sns.barplot(data=agg, x=lang_col, y=cov_col, ax=ax)
    ax.set_title("RQ2 — Mean coverage by language")
    ax.set_xlabel("Language")
    ax.set_ylabel("Mean coverage")
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()

    save_fig(fig, "rq2_mean_coverage_by_language")
    plt.show()

## 6. Analyse de Sévérité

Cette cellule enregistre un snapshot CSV des données analysées pour faciliter le reporting hors-notebook (tableaux, figures, annexes), tout en gardant une trace horodatée.

In [ ]:
# Snapshot CSV
# - Exporte un instantané des données (utile pour annexes / debug / reproductibilité).

# Save a compact CSV snapshot for later reporting (optional)
snapshot_path = RESULTS_DIR / f"rq2_inconsistency_snapshot_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df.to_csv(snapshot_path, index=False)
print("✓ Snapshot saved:", snapshot_path)

## 7. Statistiques Descriptives

Cette cellule calcule des statistiques descriptives (par modèle) et quelques mesures simples (ex. corrélation) pour résumer quantitativement les résultats RQ2 avant la conclusion.

In [ ]:
# Statistiques descriptives
# - Fournit un résumé par modèle (describe) et une corrélation simple.
# - Donne aussi une vue des couvertures moyennes (Java/Gherkin) par modèle.

print("📊 Statistiques descriptives (réel)\n")
print("=" * 60)

if results_df.empty:
    print("⚠ Pas de données.")
else:
    # Résumé cohérence par modèle
    by_model = results_df.groupby('llm_model')['coherence_score'].describe()[['count','mean','std','min','50%','max']]
    print("\n1. Cohérence par modèle:")
    display(by_model)

    # Corrélation cohérence vs inconsistance (déjà affichée en 5.4)
    corr = results_df[['total_inconsistencies', 'coherence_score']].corr().iloc[0, 1]
    print(f"\n2. Corrélation Pearson (inconsistances vs cohérence): {corr:.4f}")

    # Ratios de couverture
    cov = results_df.groupby('llm_model')[['java_coverage_ratio','gherkin_coverage_ratio']].mean()
    print("\n3. Couverture moyenne (Java/Gherkin) par modèle:")
    display(cov)

print("\n" + "=" * 60)

## 8. Conclusions et Recommandations

Cette cellule produit un résumé “lecture rapide” des résultats (meilleur modèle, seuil, volumes d’incohérences) afin de conclure l’analyse RQ2 et préparer l’écriture de la section résultats.

In [ ]:
# Conclusion (résumé exécutable)
# - Affiche une synthèse par modèle et identifie le meilleur modèle selon la cohérence moyenne.
# - Recalcule `passes_threshold` si un seuil est configuré.

print("📋 RÉSUMÉ RQ2: Validation de Cohérence Oracle ↔ Tests\n")
print("=" * 60)

if results_df.empty:
    print("⚠ Rapport sans endpoint_results: rien à résumer.")
else:
    cfg = report.get('config', {})
    thresholds = cfg.get('thresholds', {})
    min_coherence = thresholds.get('min_coherence_score')
    if min_coherence is not None:
        results_df['passes_threshold'] = results_df['coherence_score'] >= float(min_coherence)

    print(f"\n🎯 Modèles évalués: {sorted(results_df['llm_model'].unique().tolist())}")
    print(f"   Endpoints évalués: {len(results_df)}")
    print(f"   Seuil cohérence: {min_coherence}")

    summary = (
        results_df.groupby('llm_model')
        .agg(
            coherence_mean=('coherence_score', 'mean'),
            pass_rate=('passes_threshold', 'mean'),
            critical_total=('critical_count', 'sum'),
            major_total=('major_count', 'sum'),
            minor_total=('minor_count', 'sum'),
        )
        .reset_index()
        .sort_values('coherence_mean', ascending=False)
    )
    print("\n📊 Synthèse par modèle:")
    display(summary)

    best = summary.iloc[0]
    print(f"\n🏆 Meilleur modèle (cohérence moyenne): {best['llm_model']} ({best['coherence_mean']:.3f})")

print("\n" + "=" * 60)

## 9. Export des Résultats

Cette cellule exporte les tables finales (CSV) et un résumé JSON reproductible pour conserver les résultats de l’analyse et permettre leur réutilisation dans les scripts de reporting.

In [ ]:
# Exports (CSV + résumé JSON)
# - Exporte les tables principales (par endpoint + détail incohérences).
# - Produit un résumé JSON compact réutilisable dans le reporting.

# Exports (réel)
export_dir = RESULTS_DIR
export_dir.mkdir(parents=True, exist_ok=True)

# Export des tables principales
results_csv = export_dir / 'rq2_endpoint_results.csv'
incs_csv = export_dir / 'rq2_inconsistencies_detailed.csv'
results_df.to_csv(results_csv, index=False)
inconsistencies_df.to_csv(incs_csv, index=False)
print(f"✓ Endpoint results exportés: {results_csv}")
print(f"✓ Inconsistances détaillées exportées: {incs_csv}")

# Export d'un résumé JSON
cfg = report.get('config', {})
thresholds = cfg.get('thresholds', {})
min_coherence = thresholds.get('min_coherence_score')
if not results_df.empty and min_coherence is not None:
    results_df['passes_threshold'] = results_df['coherence_score'] >= float(min_coherence)

summary_by_model = {}
if not results_df.empty:
    grp = results_df.groupby('llm_model')
    for model, dfm in grp:
        summary_by_model[model] = {
            'endpoints': int(len(dfm)),
            'coherence_mean': float(dfm['coherence_score'].mean()),
            'pass_rate': float(dfm['passes_threshold'].mean()) if 'passes_threshold' in dfm.columns else None,
            'critical_total': int(dfm['critical_count'].sum()),
            'major_total': int(dfm['major_count'].sum()),
            'minor_total': int(dfm['minor_count'].sum()),
            'java_coverage_mean': float(dfm['java_coverage_ratio'].mean()),
            'gherkin_coverage_mean': float(dfm['gherkin_coverage_ratio'].mean()),
        }

out_json = export_dir / 'rq2_analysis_summary.json'
payload = {
    'source_report': str(REPORT_PATH),
    'experiment_id': report.get('experiment_id'),
    'generated_at': datetime.now().isoformat(),
    'config': cfg,
    'summary_by_model': summary_by_model,
}
with open(out_json, 'w') as f:
    json.dump(payload, f, indent=2)
print(f"✓ Résumé exporté: {out_json}")

print("\n✅ Analyse RQ2 (réel) terminée")